# 03 Build Silver Tables

Transform Bronze MMSDM rows into typed Silver tables at stable grains for price, demand, regional dispatch, interconnector flows, and generation where source columns are available.

## Configure Silver Run

This cell detects local versus Fabric runtime and defines run parameters. Spark transformations run only in Fabric.

In [1]:
# Cell purpose: Configure Silver Transformation Run.
from pathlib import Path
import importlib
import os
import sys
import uuid

try:
    spark
    is_local_run = False
except NameError:
    is_local_run = True

run_id = str(uuid.uuid4())
print(f"run_id={run_id}")
print(f"runtime={'local' if is_local_run else 'fabric'}")

run_id=3d8bd0e2-0ea5-461a-8086-32b5096f0fb9
runtime=local


## Resolve Runtime Paths and Imports

This cell resolves package paths and imports Spark helpers only in Fabric.

In [2]:
# Cell purpose: Resolve Runtime Paths and Imports.
def find_repo_root(start: Path) -> Path:
    """Find the local repo root from a notebook working directory."""

    for candidate in (start, *start.parents):
        if (candidate / "src" / "nem_fabric").exists():
            return candidate
    return start


repo_root = None
local_output_root = None
package_paths = []
# Cell purpose: Build Silver price and demand tables.
if is_local_run:
    repo_root = find_repo_root(Path.cwd())
    local_output_root = repo_root / "data"
    package_paths.append(repo_root / "src")
else:
    package_paths.append(
        Path(os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs"))
    )

for package_path in package_paths:
    if package_path.exists() and str(package_path) not in sys.path:
        sys.path.insert(0, str(package_path))

# Cell purpose: Build Silver interconnector flow table.
if is_local_run:
    import pandas as pd

    from nem_fabric.common_transformations import (
        build_silver_generation_by_unit,
        build_silver_interconnector_flows,
        build_silver_price_demand,
    )
    from nem_fabric.local_ingestion import append_csv_rows
else:
    F = importlib.import_module("pyspark.sql.functions")
    Window = importlib.import_module("pyspark.sql.window").Window

print("Python search paths added:", [str(path) for path in package_paths if path.exists()])
# Cell purpose: Build Silver generation-by-unit table when source columns are available.
if is_local_run:
    print(f"Local output root: {local_output_root}")

Python search paths added: ['c:\\Users\\brcol\\My Drive\\Documents\\!!!Resume\\Sample Work\\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\\src']
Local output root: c:\Users\brcol\My Drive\Documents\!!!Resume\Sample Work\Fabric - NEMWeb Energy Market Data Pipeline (Lakehouse, PySpark, Delta, Power BI, Direct Lake)\data


## Define Shared Silver Helpers

This cell defines NEM region mapping, table checks, and region-name enrichment. It also confirms that Bronze data exists.

In [3]:
# Cell purpose: Define Shared Silver Helpers.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    bronze_path = local_output_root / "tables" / "nem_bronze_mmsdm_rows.csv"
    if not bronze_path.exists():
        raise RuntimeError("Local Bronze CSV does not exist. Run notebook 02 first.")
    bronze_pdf = pd.read_csv(bronze_path, dtype=str).fillna("")
    print(f"Local Bronze rows loaded: {len(bronze_pdf)}")
else:
    REGION_MAP = {
        "QLD1": "Queensland",
        "NSW1": "New South Wales",
        "VIC1": "Victoria",
        "SA1": "South Australia",
        "TAS1": "Tasmania",
    }


    def table_exists(table_name: str) -> bool:
        """Return True when a Lakehouse table exists in the current Spark catalogue."""
        return spark.catalog.tableExists(table_name)


    def with_region_name(df, region_col="region"):
        """Add a business-friendly region name using Spark expressions."""
        mapping_expr = F.create_map([item for pair in REGION_MAP.items() for item in (F.lit(pair[0]), F.lit(pair[1]))])
        return df.withColumn("region_name", mapping_expr[F.col(region_col)])


    if not table_exists("nem_bronze_mmsdm_rows"):
        raise RuntimeError("nem_bronze_mmsdm_rows does not exist. Run notebook 02 first.")

    bronze_sdf = spark.table("nem_bronze_mmsdm_rows")

Local Bronze rows loaded: 254382


## Build Silver Price and Demand

This cell joins Dispatch `PRICE` and `REGIONSUM` rows into the main 5-minute regional grain and writes the core Silver tables.

In [4]:
# Cell purpose: Build Silver price and demand tables.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    silver_price_demand = build_silver_price_demand(bronze_pdf, run_id=run_id)
    if silver_price_demand.empty:
        print("No local Silver price/demand rows produced.")
    else:
        records = silver_price_demand.astype(str).to_dict("records")
        append_csv_rows(local_output_root / "tables" / "nem_silver_price_demand_5min.csv", records)
        append_csv_rows(local_output_root / "tables" / "nem_silver_regional_dispatch.csv", records)
        print(f"Local Silver price/demand rows written: {len(silver_price_demand)}")
else:
    # PRICE and REGIONSUM rows share interval and region keys. Joining them creates the main 5-minute price/demand grain.
    price = (
        bronze_sdf.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "PRICE"))
        .select(
            F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
            F.upper(F.col("regionid")).alias("region"),
            F.col("intervention").cast("int").alias("intervention"),
            F.col("rrp").cast("double").alias("price_aud_mwh"),
            F.col("source_url"),
            F.col("source_zip_name"),
            F.col("row_hash").alias("price_row_hash"),
        )
    )

    regionsum = (
        bronze_sdf.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "REGIONSUM"))
        .select(
            F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
            F.upper(F.col("regionid")).alias("region"),
            F.col("intervention").cast("int").alias("intervention"),
            F.col("totaldemand").cast("double").alias("demand_mw"),
            F.col("availablegeneration").cast("double").alias("available_generation_mw"),
            F.col("availableload").cast("double").alias("available_load_mw"),
            F.col("demandforecast").cast("double").alias("demand_forecast_mw"),
            F.col("dispatchablegeneration").cast("double").alias("dispatchable_generation_mw"),
            F.col("dispatchableload").cast("double").alias("dispatchable_load_mw"),
            F.col("netinterchange").cast("double").alias("net_interchange_mw"),
            F.col("excessgeneration").cast("double").alias("excess_generation_mw"),
            F.col("clearedsupply").cast("double").alias("dashboard_demand_mw"),
            F.col("semischedule_clearedmw").cast("double").alias("semi_scheduled_generation_mw"),
            (
                F.col("dispatchablegeneration").cast("double")
                - F.col("semischedule_clearedmw").cast("double")
            ).alias("scheduled_generation_mw"),
            F.col("dispatchablegeneration").cast("double").alias("dashboard_generation_mw"),
            F.col("row_hash").alias("regionsum_row_hash"),
        )
    )

    silver_price_demand = price.join(regionsum, ["settlement_datetime", "region", "intervention"], "left")
    silver_price_demand = with_region_name(silver_price_demand)
    silver_price_demand = (
        silver_price_demand
        .withColumn("trading_date", F.to_date("settlement_datetime"))
        .withColumn("year", F.year("settlement_datetime"))
        .withColumn("month", F.month("settlement_datetime"))
        .withColumn("day", F.dayofmonth("settlement_datetime"))
        .withColumn("interval_hour", F.hour("settlement_datetime"))
        .withColumn("interval_minute", F.minute("settlement_datetime"))
        .withColumn("silver_loaded_datetime", F.current_timestamp())
        .withColumn("run_id", F.lit(run_id))
        .dropDuplicates(["settlement_datetime", "region", "intervention"])
    )

    silver_price_demand.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_price_demand_5min")
    silver_price_demand.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_regional_dispatch")
    display(silver_price_demand.orderBy(F.col("settlement_datetime").desc()).limit(20))

Local Silver price/demand rows written: 1135


## Build Silver Interconnector Flows

This cell extracts interconnector flow records at interval and interconnector grain when Dispatch interconnector rows are available.

In [5]:
# Cell purpose: Build Silver interconnector flow table.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    interconnector = build_silver_interconnector_flows(bronze_pdf, run_id=run_id)
    if interconnector.empty:
        print("No local interconnector rows available.")
    else:
        append_csv_rows(
            local_output_root / "tables" / "nem_silver_interconnector_flows.csv",
            interconnector.astype(str).to_dict("records"),
        )
        print(f"Local Silver interconnector rows written: {len(interconnector)}")
else:
    # Interconnector grain: settlement interval x interconnector x intervention.
    interconnector = (
        bronze_sdf.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "INTERCONNECTORRES"))
        .select(
            F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
            F.col("interconnectorid").alias("interconnector_id"),
            F.col("intervention").cast("int").alias("intervention"),
            F.col("meteredmwflow").cast("double").alias("metered_flow_mw"),
            F.col("mwflow").cast("double").alias("flow_mw"),
            F.col("mwlosses").cast("double").alias("losses_mw"),
            F.col("marginalvalue").cast("double").alias("marginal_value"),
            F.col("exportlimit").cast("double").alias("export_limit_mw"),
            F.col("importlimit").cast("double").alias("import_limit_mw"),
            F.to_date(F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")).alias("trading_date"),
            F.current_timestamp().alias("silver_loaded_datetime"),
            F.lit(run_id).alias("run_id"),
        )
        .dropDuplicates(["settlement_datetime", "interconnector_id", "intervention"])
    )

    if interconnector.limit(1).count() > 0:
        interconnector.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_interconnector_flows")
        display(interconnector.orderBy(F.col("settlement_datetime").desc()).limit(20))
    else:
        print("No interconnector rows available.")

Local Silver interconnector rows written: 1362


## Build Optional Silver Generation

This cell creates a generation-by-unit Silver table only if suitable DUID and generation columns exist in the Bronze source data.

In [6]:
# Cell purpose: Build Silver generation-by-unit table when source columns are available.
if is_local_run:
    if local_output_root is None:
        raise RuntimeError("local_output_root was not resolved for local run.")
    generation = build_silver_generation_by_unit(bronze_pdf, run_id=run_id)
    if generation.empty:
        print("Generation-by-unit columns not available in current local Bronze data.")
    else:
        append_csv_rows(
            local_output_root / "tables" / "nem_silver_generation_by_unit.csv",
            generation.astype(str).to_dict("records"),
        )
        print(f"Local Silver generation rows written: {len(generation)}")
else:
    # Unit generation is source-dependent. This creates a Silver table only when DUID and generation-like columns are present.
    candidate_columns = set(bronze_sdf.columns)
    if {"duid", "dispatchablegeneration"}.issubset(candidate_columns):
        generation = (
            bronze_sdf.filter(F.col("duid") != "")
            .select(
                F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
                F.col("duid"),
                F.col("dispatchablegeneration").cast("double").alias("generation_mw"),
                F.to_date(F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")).alias("trading_date"),
                F.current_timestamp().alias("silver_loaded_datetime"),
                F.lit(run_id).alias("run_id"),
            )
            .dropDuplicates(["settlement_datetime", "duid"])
        )
        generation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_generation_by_unit")
    else:
        print("Generation-by-unit columns not available in current Bronze data.")

Local Silver generation rows written: 12678
